# Module 2 — NumPy arrays and plotting signals

**Who this is for:** you know basic Python but have not used NumPy or matplotlib much.

> **Self-check — skip this module if you can already:**
>
> - [ ] build a time axis from a sampling rate and plot a signal with labelled axes in seconds
> - [ ] explain what `x.shape` is for a 4-channel, 10-second recording at 256 Hz
> - [ ] select the samples between 2 s and 3 s without a loop
> - [ ] compute a statistic per channel with `axis=`
>
> If you can tick every box, go to the next module. If not, work through the parts you can't tick. Before
> jumping to a section, run the module's **setup cell** (the first code cell): later sections use variables
> defined earlier, so run the cells above a section if you get a `NameError`.

**You will learn**
- arrays vs lists, `dtype` and `shape`
- the time axis: `t = np.arange(n) / fs`
- slicing, boolean masks and vectorised maths
- multi-channel arrays and `axis`
- reproducible random numbers
- matplotlib figures you can read

**Where the course uses this:** every demo and lab. Chapter 1–2 demos and Labs 1–2 lean on this most.

*Run each cell with **Shift+Enter**, top to bottom. Nothing here needs data files; once the libraries are
installed (see the [overview](README.md#setup-first)), the notebook runs offline.*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/python_tutorial/02_numpy_and_plotting.ipynb)
&nbsp; Part of the **Python for biosignal processing** tutorial — see the [overview](README.md) to choose which modules you need.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Arrays instead of lists

A NumPy **array** holds many numbers of the same type and does maths on all of them at once
("vectorised"). Signals are arrays.

In [ ]:
rr_list = [0.82, 0.80, 0.85, 0.79]
rr = np.array(rr_list)
print(60 / rr)                  # works on every element: no loop needed
print(rr.dtype, rr.shape, rr.size, rr.mean(), rr.std())
# with a plain list, 60 / rr_list raises a TypeError

## 2. The time axis

A sampled signal is just numbers. The **sampling rate** `fs` (samples per second) is what turns sample
indices into time. Always keep `fs` next to the samples.

In [ ]:
fs = 250                                   # Hz
duration_s = 4.0
n = int(fs * duration_s)
t = np.arange(n) / fs                      # 0, 1/fs, 2/fs, ...  in seconds
print(t[:5], "...", t[-1])

rng = np.random.default_rng(2013)          # a SEEDED random generator: same numbers every run
x = 1.0 * np.sin(2 * np.pi * 1.2 * t) + 0.3 * np.sin(2 * np.pi * 10 * t) + 0.1 * rng.standard_normal(n)
print(x.shape)

**Use `np.random.default_rng(seed)`** for random numbers. With a fixed seed you get the same numbers
every time, which is what makes results repeatable (the capstone grades this).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(t, x, lw=0.8)
ax.set_xlabel("time (s)")
ax.set_ylabel("amplitude (a.u.)")
ax.set_title("1.2 Hz rhythm + 10 Hz component + noise")
ax.grid(True)
plt.show()

A readable plot has **labelled axes with units**, a title, and a sensible time range. Use
`fig, ax = plt.subplots(...)` and call methods on `ax` — every course notebook plots this way.

## 3. Slicing and boolean masks

Select by **index** (`x[a:b]`) or by **condition** (a boolean mask). Both avoid loops.

In [ ]:
seg = x[int(2.0 * fs):int(3.0 * fs)]       # samples from 2 s up to (not including) 3 s
print(seg.shape)                            # 250 samples = 1 s

mask = (t >= 2.0) & (t < 3.0)               # same selection by time; use & and | with parentheses
print(np.array_equal(x[mask], seg))

above = x > 1.0                             # where is the signal above 1?
if above.any():                             # t[above][0] would fail if there is no crossing at all
    print(f"{above.mean():.1%} of samples are above 1.0; first at t = {t[above][0]:.3f} s")
else:
    print("the signal never goes above 1.0")

## 4. Multi-channel recordings and `axis`

A recording with several channels is a 2-D array. The course (and MNE) use **channels × samples**:
`x.shape == (n_channels, n_samples)`. `axis=1` means "along time", `axis=0` means "across channels".

In [ ]:
n_ch = 4
X = 0.5 * rng.standard_normal((n_ch, n)) + np.sin(2 * np.pi * 1.2 * t)   # broadcasting: (4, n) + (n,)
print(X.shape)

rms_per_channel = np.sqrt(np.mean(X ** 2, axis=1))      # one value per channel
mean_over_channels = X.mean(axis=0)                     # one value per sample
print(rms_per_channel.round(3), mean_over_channels.shape)

fig, axes = plt.subplots(n_ch, 1, figsize=(9, 5), sharex=True)
for k, ax in enumerate(axes):
    ax.plot(t, X[k], lw=0.6)
    ax.set_ylabel(f"ch {k}")
axes[-1].set_xlabel("time (s)")
plt.tight_layout(); plt.show()

**Broadcasting** lets arrays of different shapes combine. NumPy lines the shapes up from the right, and each
pair of dimensions must be **equal, or one of them must be 1**. So `(4, 1000) + (1000,)` works, and
`(4, 1000) + (4,)` does not (1000 vs 4) — use `offsets[:, None]` to make it `(4, 1)`, which then stretches
along time. When you see *"operands could not be broadcast"*, print the shapes.

In [ ]:
offsets = np.array([0.0, 1.0, 2.0, 3.0])
X_shifted = X - offsets[:, None]             # subtract one offset per channel
print(X_shifted.shape)

## 5. Epochs: cutting a recording into windows

Many tasks label fixed windows (a 30-s sleep epoch, a 200-ms EMG window). Reshape instead of looping:

In [ ]:
win_s = 0.5
w = int(win_s * fs)
n_win = x.size // w
epochs = x[:n_win * w].reshape(n_win, w)     # (n_windows, samples_per_window)
print(epochs.shape, "-> per-window variance:", epochs.var(axis=1).round(3))
print(f"{x.size - n_win * w} samples at the end did not fill a whole window and were dropped")

## 6. Loops vs vectorised code

Both give the same answer, but vectorised NumPy is far faster on real recordings (millions of samples):

In [ ]:
import time
big = rng.standard_normal(2_000_000)

t0 = time.perf_counter(); s = 0.0
for v in big:
    s += v * v
loop_s = time.perf_counter() - t0

t0 = time.perf_counter(); s2 = np.sum(big ** 2); vec_s = time.perf_counter() - t0
print(f"loop {loop_s:.3f} s   numpy {vec_s:.4f} s   same answer: {np.isclose(s, s2)}")

## Try it yourself

For the 4-channel array `X` above: find the channel with the **largest peak-to-peak amplitude**
(`max - min` along time) and the **time in seconds** of that channel's maximum.

In [ ]:
# TODO: use np.ptp(X, axis=...) or X.max(axis=...) - X.min(axis=...), then np.argmax
best_ch = ...
t_of_max = ...

In [ ]:
# Check: keep this cell separate, so it survives when you paste in a solution.
if best_ch is ... or t_of_max is ...:
    print("Not done yet: set best_ch and t_of_max above.")
elif isinstance(best_ch, (bool, np.bool_)) or not isinstance(best_ch, (int, np.integer)):
    print(f"not yet: best_ch must be an integer channel index, got {best_ch!r}")
elif not (0 <= best_ch < X.shape[0]) or not (0 <= float(t_of_max) <= t[-1]):
    print(f"not yet: best_ch must be 0..{X.shape[0] - 1} and t_of_max between 0 and {t[-1]:.3f} s")
else:
    ptp = X.max(axis=1) - X.min(axis=1)
    k = int(round(float(t_of_max) * fs))                     # sample index of the time you gave
    ok_ch = ptp[int(best_ch)] == ptp.max()
    ok_t = np.isclose(t[k], t_of_max) and X[int(best_ch), k] == X[int(best_ch)].max()
    print(f"channel: {'correct' if ok_ch else 'not yet'}   time of maximum: {'correct' if ok_t else 'not yet'}")

**Solution:** try it yourself first, then compare with the [solutions notebook](solutions.ipynb) (Module 2). Yours may differ and still be right.

**Next:** Module 3 (how to store signals with their metadata) or Module 4 (signal processing).